In [5]:
# 装饰器
'''
装饰器（decorator）是 Python 中的一种高级功能，用于在不修改原函数代码的前提下，动态扩展函数或类的功能。
本质上，装饰器是一个函数：它接收一个函数作为参数，并返回一个新的函数（通常是对原函数的增强版本）。

装饰器通过 @decorator_name 语法应用在函数或方法定义之前。
Python 还提供了一些内置装饰器，例如 @staticmethod 和 @classmethod。
常见应用场景：
日志记录：记录函数调用信息、参数和返回值
性能统计：统计函数执行时间
权限控制：限制函数访问权限
缓存：缓存函数结果，提高性能

基本语法
装饰器的核心思想是：用一个函数"包装"另一个函数。
Python 的函数可以像数字、字符串一样赋值、传递、返回，这是装饰器能实现的根本。
'''

def add(x, y):
    return x + y

# 包装器：接收原函数，返回新函数
def wrapper(func):
    # 内部新函数，负责增加逻辑+执行原函数
    def new_func(a, b):
        print("====函数开始====")
        res = func(a, b)  # 执行原始函数
        print("====函数结束====")
        return res  # 把原始函数的计算结果返回给外部调用者。
    # 返回内部函数对象，不调用（只返回，不执行）
    return new_func

# 包装原add函数；手动装饰：覆盖add变量
add = wrapper(add)
# 现在调用的是增强后的新函数
print(add(1, 2))    
'''
输出结果：
====函数开始====
====函数结束====
3 

赋值前后对比
赋值前：add = 原始求和函数
赋值后：add = 包装后的 new_func 函数

add = wrapper(add)
步骤 1：执行右边 wrapper(add)
找到一开始定义的原始函数 add（求和函数）；
把这个原始函数对象作为实参传给 wrapper 的形参 func；
func，即这个修饰函数的最外层的参数只是起到一个传递被修饰函数 add 的作用；
而内部的 new_func 才是修饰的核心内容，内层函数的返回值是执行修饰目的后的结果
外层的返回值，则是传递内层修饰函数
等价：func = 原始add函数
进入 wrapper 函数内部，定义 new_func；
执行 return new_func，把内层包装函数对象吐出来。
步骤 2：= 赋值覆盖左边变量 add
把上面返回的 new_func 对象，赋值给变量 add。
'''

# 装饰器（包装器工厂）
# 这个等价于上面那段代码
def wrapper(func):
    def new_func(a, b):
        print("====函数开始====")
        res = func(a, b)
        print("====函数结束====")
        return res
    return new_func

# @wrapper 等价于 add = wrapper(add)
@wrapper
def add(x, y):
    return x + y

print(add(1,2))      
'''
输出结果：
====函数开始====
====函数结束====
3 
'''

'''
注意：
new_func：代表函数本身（函数对象）
new_func()：代表执行这个函数，拿到返回值
两者区别较大
new_func 和 func () 核心区别
1. new_func：代表函数对象本身
不加括号：只是拿到函数这个变量 / 内存地址，不会执行函数
打印输出：显示函数内存地址、函数名、定义位置
用途：把函数当参数传递、赋值、装饰器传参
2. func()：调用 / 执行函数
加括号 = 触发函数内部代码运行，得到函数返回值
打印输出：打印函数 return 的结果
必须满足参数要求，缺参会直接报错
'''

====函数开始====
====函数结束====
3
====函数开始====
====函数结束====
3


'\n====函数开始====\n====函数结束====\n3 '

In [9]:
# 带参数的装饰器
# 场景：装饰器本身需要传入配置参数，比如自定义打印文字、设置重试次数。
# 如果原函数有参数，需要在 wrapper 中使用 *args, **kwargs：
# 外层函数：接收装饰器参数，返回真正的装饰器
def log_prefix(prefix):
    # 真正的装饰器：接收函数
    def decorator(func):
        def wrapper(*args, **kwargs):
            print(f"[{prefix}] 执行函数：{func.__name__}")
            res = func(*args, **kwargs)
            print(f"[{prefix}] 执行完毕")
            return res
        return wrapper
    return decorator

# 调用log_prefix("INFO")，返回decorator，等价于 @decorator
@log_prefix("INFO")
def add(x,y):
    return x+y

add(10,20)

# 这里装饰器的参数起到一个改变打印内容的作用

'''
func.__name__

__name__ 是 Python 内置特殊属性（双下划线属性，叫魔法属性）
所有函数对象天生自带 .__name__，是 Python 解释器自动给函数生成的内置属性，不用自己定义。
作用
存储这个函数定义时的原始函数名字符串。
'''

'''
带参数装饰器（以这儿的三层函数为例子）
结构：
① 最外层：接收自定义配置参数；
② 中层：标准装饰器，接收目标函数 func；
③ 内层：万能包装函数 wrapper(*args, **kwargs)，接管原函数调用。
写法：@装饰器名(配置参数)
****特点：依靠外层传入参数，灵活修改装饰行为。

写在 @xxx (这里) 的参数，就是最外层函数的参数，二者完全等价。
@装饰器(实参) 里所有实参，全部交给第一层外层函数接收。
两处的参数数量和形式要相对应

带参装饰器外层参数作用
作为配置项，通过闭包永久保存到内层，控制装饰内部逻辑，实现一套代码多种效果。
常见配置：日志前缀、重试次数、权限列表、缓存过期时间、功能开关等。


什么时候用带参装饰器
同一套装饰逻辑，但不同函数需要差异化设置时，必须三层带参装饰器：
自定义日志标识、打印文案；
接口重试、缓存自定义时长；
权限校验，区分可访问角色；
功能开关，按需开启 / 关闭装饰逻辑。（常用布尔值作为参数）
'''

[INFO] 执行函数：add
[INFO] 执行完毕


30

In [6]:
# 多个装饰器叠加执行顺序
# 多个装饰器在定义阶段从下到上依次包裹函数，调用阶段从上到下依次执行：
def decorator1(func):
    def wrapper():
        print("装饰器1 前置")
        func()
        print("装饰器1 后置")
    return wrapper

def decorator2(func):
    def wrapper():
        print("装饰器2 前置")
        func()
        print("装饰器2 后置")
    return wrapper

# 叠加装饰器，先执行下面的decorator2，再decorator1
@decorator1
@decorator2
def test():
    print("原始函数运行")

test()

'''
包装顺序是就近原则，离他最近的先包装然后依次往后，即从下到上
执行顺序是先执行最外层的包装结构，即从上到下
'''

装饰器1 前置
装饰器2 前置
原始函数运行
装饰器2 后置
装饰器1 后置


In [10]:
# 类装饰器
'''
除了函数，装饰器也可以作用于类。
类装饰器接收一个类，并返回修改后的类或包装类。

增强类方法
控制实例化过程
实现单例、日志等功能
'''

# 函数形式的类装饰器
def log_class(cls):
    # cls = 原始类 MyClass
    class Wrapper:
        def __init__(self,*args,**kwargs):
            # 创建原生MyClass实例，存到self.wrapped
            self.wrapped = cls(*args,**kwargs)

        def __getattr__(self,name):
            return getattr(self.wrapped,name)

        def display(self):   # 具体的执行函数
            print("调用前")
            self.wrapped.display()   # 执行原生类的display
            print("调用后")

        ''' # 新增 __call__，支持 obj()
        def __call__(self):
            # 你想执行什么逻辑就写在这里
            print("直接调用obj()触发__call__")
            self.display() '''
        
    return  Wrapper   # 用新Wrapper类替换原来的MyClass

@log_class
class MyClass:
    def display(self):
        print("原方法")

obj = MyClass()
obj.display()    #默认情况下 obj() 不能直接调用，因为只有在装饰类里实现了 __call__ 魔法方法的类实例，才支持像函数一样 obj() 执行；
                 # 普通类只有 obj.方法名() 这种调用形式。
                 # 不加自定义 __call__ → 只能 obj.display()，不能 obj()；

'''
普通装饰器装饰函数，这个装饰器 log_class 专门装饰类：
@log_class 作用在类 MyClass 上；
语法糖等价：MyClass = log_class(MyClass)；
log_class 接收参数 cls，cls 就是原始类 MyClass；
装饰器内部定义新包装类 Wrapper，最后返回这个 Wrapper；
之后你写 MyClass() 创建实例，实际创建的是 Wrapper 的实例，不再是原生 MyClass。

Wrapper 的 init 构造方法
当你 obj = MyClass()：
实际运行 Wrapper()，进入 Wrapper.__init__
内部自动实例化原始类，把原生对象存在 self.wrapped，相当于持有被包装对象。

__init__（双下划线，Python 魔法方法）
全称：initialize，初始化
含义
类实例被创建时自动执行的构造方法；
作用：给实例绑定属性、初始化数据、创建依赖对象、传参赋值。
行业通用理解
所有面向对象语言里，init 系列都是「初始化」；
Python 强制固定名 __init__，不能改名；

getattr 魔法方法（代理所有原类属性 / 方法）
作用：
访问 obj.xxx 时，如果 Wrapper 自身没有 xxx 属性，就会触发 __getattr__，自动转发给内部原生对象 self.wrapped。
比如原类有普通方法、属性，obj.属性 能正常使用，不会报错。

重写同名 display 方法（核心增强逻辑）
Wrapper 自己定义了 display，优先级高于 getattr：
调用 obj.display() 会直接走这个包装方法，在原生方法前后加自定义打印逻辑，实现功能增强。

完整执行流程 obj = MyClass(); obj.display()
obj = MyClass() → 创建 Wrapper 实例
执行 Wrapper.__init__
self.wrapped = MyClass() 生成原生对象
obj.display()
Wrapper 自带 display，不走 getattr
打印 调用前
self.wrapped.display() 执行原生类的 print("原方法")
打印 调用后

魔法方法（前后双下划线 xxx，Python 内置约定）
__init__：初始化构造
__getattr__：访问不存在属性时自动触发，多用于代理、包装类（你类装饰器里用到）
__name__：函数 / 类内置属性，存储定义时的名字
__call__：让实例可以像函数一样 obj() 调用
__str__：print (对象) 时返回可读字符串

装饰类里的三个函数都有相同的参数 self：
self 到底是什么
self 代表当前这个类的实例对象本身。
当你调用 obj.display()，Python 会自动把 obj 这个实例，作为第一个参数传给方法，这个参数约定命名为 self。

三个全是实例方法，遵守同一个规则：
第一个形参固定接收调用者实例；
约定名称统一用 self（只是约定，换成 a、obj 都行，但行业全部用 self）；
有了 self，才能访问当前实例身上绑定的属性，比如 self.wrapped。

什么时候要 self、什么时候不用
只要是不带装饰器（@classmethod/@staticmethod）的类内函数，第一个参数：约定 cls，代表类本身；其余全是实例方法，强制第一个参数写 self；
self 是实例的代称，用来读取 / 修改当前对象身上存储的属性；
self 只是行业统一命名规范，不是关键字，换成别的名字代码也能跑，但所有人都约定用 self。
'''

调用前
原方法
调用后


In [16]:
# 类形式的类装饰器
class ClassDecorator:
    def __init__(self, cls):   # 初始化
        self.cls = cls
        self.instance = None  # 用来缓存唯一实例，初始为空

    def __call__(self, *args, **kwargs):
        if self.instance is None:
            self.instance = self.cls(*args, **kwargs)
        return self.instance


# 装饰业务类
@ClassDecorator
class MyClass:
    def display(self):
        print("原生方法")

obj = MyClass()  # 触发 __call__
obj.display()
'''
obj = MyClass()
所有 class 定义出来的类型，底层属于可调用类型，Python 语法规定：
只要是「可调用对象」，就能用 xxx() 这种函数式调用语法。
可调用对象包含这几类：
普通函数 def func():
类 class MyClass:
实现了 __call__ 的类实例（前面讲的类装饰器实例）

所有 class 定义的类，本身自带可调用能力，类名() 是创建实例的标准语法；
类名() = 调用类，触发 __init__，返回实例；
实例对象默认不能 obj()，只有实现 __call__ 才能括号调用；
类 ≠ 实例：类是模板（可调用），实例是成品（默认不可调用）。
'''
'''
@ClassDecorator → MyClass = ClassDecorator(MyClass)
__init__ 接收原始类 MyClass，存到 self.cls
obj = MyClass() → 等价 装饰器实例()，触发 __call__
在 __call__ 内部：实例化原始类、增强方法、返回改造后的实例
'''

原生方法


'\n@ClassDecorator → MyClass = ClassDecorator(MyClass)\n__init__ 接收原始类 MyClass，存到 self.cls\nobj = MyClass() → 等价 装饰器实例()，触发 __call__\n在 __call__ 内部：实例化原始类、增强方法、返回改造后的实例\n'

In [19]:
# 内置装饰器
'''
常用内置装饰器：
@staticmethod：定义静态方法
@classmethod：定义类方法
@property：将方法变为属性
'''
class MyClass:
    def __init__(self):
        self._name = "默认名称"

    @staticmethod
    def static_method():
        print("静态方法")

    @classmethod
    def class_method(cls):
        print("类名：", cls.__name__)

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        self._name = value

# 静态方法
MyClass.static_method()

# 类方法
MyClass.class_method()

# property
obj = MyClass()
print(obj.name)       # 读属性，不用括号
obj.name = "测试名字"  # 赋值，触发setter
print(obj.name)

'''
@staticmethod 静态方法
特点
不需要 self /cls 第一个参数，没有自动传入的隐参；
和类、实例本身都无关，就是单纯放在类里的普通函数；
调用两种方式：类名.方法() / 实例.方法()；
不能访问实例属性 self.xxx，也不能访问类属性 cls.xxx。

@classmethod 类方法
特点
第一个参数固定 cls，代表类本身（MyClass）；
可以读取 / 修改类属性，不能直接操作实例独有的 self.xxx；
调用：MyClass.class_method()，自动把 MyClass 传给 cls；
常用场景：工厂方法、批量修改类静态变量。

@property 属性装饰器（只读）+ @xxx.setter（修改）
作用
把方法伪装成实例属性，不用加 () 调用，实现封装：
外部只能通过 obj.name 获取值；
@name.setter 允许 obj.name = "xxx" 赋值；
内部真实数据存在私有变量 self._name，避免直接暴露。
执行流程
读取：print(obj.name) → 触发上面 @property 的 name 方法；
赋值：obj.name = "小明" → 触发 @name.setter 方法，value = "小明"；
约定：真实底层变量带下划线 _name，外部不要直接访问 obj._name。

静态方法
类名： MyClass
默认名称
测试名字
